In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "enviroment_bj").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("project_root:", ROOT)

from copy import deepcopy

from enviroment_bj import BlackjackConfig, BlackjackEnvironment, ObservationConfig, StartStateConfig
from model.agents import RecurrentDoubleDQN
from training import (
    ReplayBufferConfig,
    EpsilonScheduleConfig,
    OptimizationConfig,
    TargetUpdateConfig,
    EvaluationConfig,
    CheckpointConfig,
    PrintConfig,
    TrainerConfig,
    TrainingPipelineConfig,
    train_model,
)


project_root: C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl


# RecurrentDoubleDQN over table_realistic_default + fresh_shoe

In [2]:
# =========================
# OBSERVATION / START STATE
# =========================

observation_config = ObservationConfig.for_profile("table_realistic_default")

start_state_config = StartStateConfig(
    mode="fresh_shoe",
    min_burned_rounds=0,
    max_burned_rounds=0,
    clear_visible_histories_after_burn=True,
    hide_reshuffle_progress_from_observation=False,
)

# =========================
# TABLE 1: S17 + surrender
# =========================

blackjack_config_1 = BlackjackConfig(
    n_decks=6,
    shoe_penetration=0.80,
    dealer_hits_soft_17=False,   # S17
    blackjack_payout=1.5,
    dealer_peeks_for_blackjack=True,
    double_allowed_on="any_two_cards",
    double_after_split_allowed=True,
    split_rule="same_value",
    max_hands_after_split=4,
    resplit_aces_allowed=True,
    hit_split_aces_allowed=False,
    surrender_allowed=True,
    insurance_allowed=True,
    base_bet=1.0,
    strict_shoe_validation=False,
    observation=deepcopy(observation_config),
    observation_mode=None,
    expose_shoe_composition=False,
)

env1 = BlackjackEnvironment(
    config=blackjack_config_1,
    seed=77,
    start_state=deepcopy(start_state_config),
)

# =========================
# TABLE 2: H17 + no surrender
# =========================

blackjack_config_2 = BlackjackConfig(
    n_decks=6,
    shoe_penetration=0.80,
    dealer_hits_soft_17=True,    # H17
    blackjack_payout=1.5,
    dealer_peeks_for_blackjack=True,
    double_allowed_on="any_two_cards",
    double_after_split_allowed=True,
    split_rule="same_value",
    max_hands_after_split=4,
    resplit_aces_allowed=True,
    hit_split_aces_allowed=False,
    surrender_allowed=False,     # cambia política
    insurance_allowed=True,
    base_bet=1.0,
    strict_shoe_validation=False,
    observation=deepcopy(observation_config),
    observation_mode=None,
    expose_shoe_composition=False,
)

env2 = BlackjackEnvironment(
    config=blackjack_config_2,
    seed=7,
    start_state=deepcopy(start_state_config),
)

envs = [env1, env2]

In [ ]:
# =========================
# MODEL
# =========================

model = RecurrentDoubleDQN.from_profile(
    "table_realistic_default",
    activation="relu",
    use_layer_norm=True,
    dropout=0.0,
    projection_dim=256,
    recurrent_hidden_dim=256,
    recurrent_num_layers=1,
    recurrent_type="gru",  
    head_hidden_dim=128,
)

# =========================
# TRAINING PIPELINE
# =========================

pipeline_config = TrainingPipelineConfig(
    trainer=TrainerConfig(
        total_epochs=60,
        env_steps_per_epoch=1_000,
        train_frequency=4,
        updates_per_train_step=1,
        max_updates_per_epoch=None,
        device="auto",
        seed=13,
        reset_hidden_on_round_end=False,
        sequence_end_on_done=False,
        flush_partial_sequences_at_epoch_end=True,
    ),
    replay_buffer=ReplayBufferConfig(
        capacity=120_000,
        batch_size=32,
        warmup_size=3_000,
        sequence_length=16,
        min_sequence_length=4,
    ),
    epsilon=EpsilonScheduleConfig(
        start=1.0,
        end=0.05,
        decay_steps=120_000,
        evaluation_epsilon=0.0,
    ),
    optimization=OptimizationConfig(
        optimizer="adam",
        learning_rate=3e-4,
        weight_decay=0.0,
        scheduler="none",
        scheduler_step_size=10_000,
        scheduler_gamma=0.99,
        gradient_clipping=True,
        max_grad_norm=5.0,
    ),
    target_update=TargetUpdateConfig(
        mode="hard",
        hard_update_interval=1_500,
        soft_tau=0.005,
    ),
    evaluation=EvaluationConfig(
        enabled=True,
        every_n_epochs=1,
        num_rounds=800,
        max_decisions=50_000,
    ),
    checkpoints=CheckpointConfig(
        directory="checkpoints/canonical_B_recurrent_two_tables",
        save_latest=True,
        save_best_eval=True,
        save_periodic=True,
        periodic_interval_updates=3_000,
        best_metric_name="ev_per_1000_hands",
        maximize_best_metric=True,
    ),
    prints=PrintConfig(
        enable=True,
        print_update_interval=100,
        print_collection_interval=500,
        print_epoch_summary=True,
        print_eval_summary=True,
        include_segment_details=False,
    ),
)

# =========================
# TRAIN
# =========================

result = train_model(
    envs=envs,
    model=model,
    pipeline_config=pipeline_config)



--------------------------------------------------------------------------------------------
Blackjack RL run | arch: recurrent | recurrent: gru | encoder: table_realistic_default | obs: table_realistic_default | start: fresh_shoe
Device: cpu | epochs: 60 | envs: 2 | steps/epoch: 2000 | updates/epoch~: 500 | params: 707,974
Optim: adam | lr: 3.00e-04 | loss: huber | gamma: 0.9900 | grad_clip: True(5.00)
Replay: warmup: 3000 | capacity: 120000 | batch: 32 | seq_len: 16 | min_seq_len: 4
Explore/Target: eps 1.000->0.050 (decay 120000) | target hard | interval 1500 | tau 0.0050
Eval/CKPT: eval_rounds: 800 | eval_decisions: 50000 | checkpoints: checkpoints\canonical_B_recurrent_two_tables
Environment: decks=6 | pen=0.80 | S17=True | payout=1.50 | double=any_two_cards | split=same_value | DAS=True
--------------------------------------------------------------------------------------------

=== Epoch 1/60 ===
[Warmup] buffer 200/3000
[Warmup] buffer 400/3000
[Warmup] buffer 600/3000
[Warmup] 

In [ ]:
print("=== TRAINING FINISHED: CANONICAL B ===")
print("Best eval metrics:")
print(result["best_eval_metrics"])
print("Checkpoint dir:")
print(result["checkpoint_dir"])

# RecurrentDoubleDQN over table_realistic_default + Unknown_shoe

In [ ]:
# =========================
# OBSERVATION / START STATE
# =========================

observation_config = ObservationConfig.for_profile("table_realistic_unknown_progress")

start_state_config = StartStateConfig(
    mode="unknown_progress",
    min_burned_rounds=5,
    max_burned_rounds=30,
    clear_visible_histories_after_burn=True,
    hide_reshuffle_progress_from_observation=True,
)

# =========================
# TWO CANONICAL TABLES
# =========================

# Mesa 1: S17 + surrender
blackjack_config_1 = BlackjackConfig(
    n_decks=6,
    shoe_penetration=0.80,
    dealer_hits_soft_17=False,   # S17
    blackjack_payout=1.5,
    dealer_peeks_for_blackjack=True,
    double_allowed_on="any_two_cards",
    double_after_split_allowed=True,
    split_rule="same_value",
    max_hands_after_split=4,
    resplit_aces_allowed=True,
    hit_split_aces_allowed=False,
    surrender_allowed=True,
    insurance_allowed=True,
    base_bet=1.0,
    strict_shoe_validation=False,
    observation=deepcopy(observation_config),
    observation_mode=None,
    expose_shoe_composition=False,
)

# Mesa 2: H17 + no surrender
blackjack_config_2 = BlackjackConfig(
    n_decks=6,
    shoe_penetration=0.80,
    dealer_hits_soft_17=True,    # H17
    blackjack_payout=1.5,
    dealer_peeks_for_blackjack=True,
    double_allowed_on="any_two_cards",
    double_after_split_allowed=True,
    split_rule="same_value",
    max_hands_after_split=4,
    resplit_aces_allowed=True,
    hit_split_aces_allowed=False,
    surrender_allowed=False,
    insurance_allowed=True,
    base_bet=1.0,
    strict_shoe_validation=False,
    observation=deepcopy(observation_config),
    observation_mode=None,
    expose_shoe_composition=False,
)

env1 = BlackjackEnvironment(
    config=blackjack_config_1,
    seed=77,
    start_state=deepcopy(start_state_config),
)

env2 = BlackjackEnvironment(
    config=blackjack_config_2,
    seed=78,
    start_state=deepcopy(start_state_config),
)

envs = [env1, env2]

In [ ]:
# =========================
# MODEL
# =========================

model = RecurrentDoubleDQN.from_profile(
    "table_realistic_unknown_progress",
    activation="relu",
    use_layer_norm=True,
    dropout=0.0,
    projection_dim=256,
    recurrent_hidden_dim=256,
    recurrent_num_layers=1,
    recurrent_type="lstm",   # recomendado aquí
    head_hidden_dim=128,
)

# =========================
# TRAINING PIPELINE
# =========================

pipeline_config = TrainingPipelineConfig(
    trainer=TrainerConfig(
        total_epochs=60,
        env_steps_per_epoch=1_000,          # ajustado para 2 mesas
        train_frequency=4,
        updates_per_train_step=1,
        max_updates_per_epoch=None,
        device="auto",
        seed=13,
        reset_hidden_on_round_end=False,
        sequence_end_on_done=False,
        flush_partial_sequences_at_epoch_end=True,
    ),
    replay_buffer=ReplayBufferConfig(
        capacity=120_000,
        batch_size=32,
        warmup_size=3_000,
        sequence_length=20,                 # un poco más larga por memoria parcial
        min_sequence_length=4,
    ),
    epsilon=EpsilonScheduleConfig(
        start=1.0,
        end=0.05,
        decay_steps=140_000,                # más lento: problema más difícil
        evaluation_epsilon=0.0,
    ),
    optimization=OptimizationConfig(
        optimizer="adam",
        learning_rate=3e-4,
        weight_decay=0.0,
        scheduler="none",
        scheduler_step_size=10_000,
        scheduler_gamma=0.99,
        gradient_clipping=True,
        max_grad_norm=5.0,
    ),
    target_update=TargetUpdateConfig(
        mode="hard",
        hard_update_interval=1_500,
        soft_tau=0.005,
    ),
    evaluation=EvaluationConfig(
        enabled=True,
        every_n_epochs=1,
        num_rounds=800,
        max_decisions=80_000,
    ),
    checkpoints=CheckpointConfig(
        directory="checkpoints/recurrent_two_tables_unknown_progress",
        save_latest=True,
        save_best_eval=True,
        save_periodic=True,
        periodic_interval_updates=3_000,
        best_metric_name="ev_per_1000_hands",
        maximize_best_metric=True,
    ),
    prints=PrintConfig(
        enable=True,
        print_update_interval=100,
        print_collection_interval=500,
        print_epoch_summary=True,
        print_eval_summary=True,
        include_segment_details=False,
    ),
)

# =========================
# TRAIN
# =========================

result = train_model(
    envs=envs,
    model=model,
    pipeline_config=pipeline_config,
)